In [ ]:
cd ..

In [ ]:
from dotenv import load_dotenv
load_dotenv(".env")

In [ ]:
# Notebook cell — JSON -> chunks -> BGE-M3 embeddings (cosine-ready) -> UPSERT into law_chunks

import os, json, re, hashlib
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import psycopg



# ── 🔧 Config minimale ──────────────────────────────────────────────────────────
JSON_PATH  = Path(os.getenv("LEGIFRANCE_JSON_PATH", "data/out/legifrance/legi_results.json"))
CODE_ID    = os.getenv("LEGIFRANCE_CODE_ID", "LEGITEXT000044416551")
THEMATIQUE = os.getenv("LEGIFRANCE_THEMATIQUE", "Droit/CGFP")
SOURCE     = os.getenv("LEGIFRANCE_SOURCE", "legifrance")

# Chunking
CHUNK_SIZE   = 1200
CHUNK_OVERLAP= 100

# Embedding
MODEL_NAME = os.getenv("EMBEDDING_MODEL", "BAAI/bge-m3")
NORMALIZE  = True                # cosine-ready vectors
EMBED_COL  = os.getenv("EMBEDDING_COLUMN", "embedding_m3")

# Connexion Postgres via env (OK avec tunnel local)
PG = dict(
    host=os.environ.get("PGHOST", "127.0.0.1"),
    port=int(os.environ.get("PGPORT", "5432")),
    user=os.environ.get("PGUSER", "postgres"),
    password=os.environ.get("PGPASSWORD", ""),
    dbname=os.environ.get("PGDATABASE", "postgres"),
)

# ── 🧩 Utilitaires ──────────────────────────────────────────────────────────────
def norm_num(num: str) -> str:
    return re.sub(r"[.\s]+", "", num or "")

def hash_id(code_id, article_id, num_article, etat, chunk_index) -> str:
    base = f"{code_id}|{article_id}|{num_article}|{etat or 'courante'}|{chunk_index}"
    return "sha1:" + hashlib.sha1(base.encode("utf-8")).hexdigest()

def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    if not text: return []
    text = re.sub(r"\s+\n", "\n", text.strip())
    chunks = []
    i = 0
    step = max(1, size - overlap)
    while i < len(text):
        chunks.append(text[i:i+size])
        i += step
    return chunks

# ── 🔢 Charger le modèle BGE-M3 ────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
model = SentenceTransformer(MODEL_NAME)  # 1024 dims pour bge-m3

# ── 📖 Lire le JSON ────────────────────────────────────────────────────────────
if not JSON_PATH.exists():
    raise FileNotFoundError(f"JSON introuvable: {JSON_PATH}")
with JSON_PATH.open("r", encoding="utf-8") as f:
    articles = json.load(f)

print(f"→ JSON chargé: {len(articles)} articles")

# ── 🧱 Construire les enregistrements (chunk + header) ─────────────────────────
records: List[Dict[str, Any]] = []
texts_for_embed: List[str] = []

for key, obj in articles.items():
    art_id     = obj.get("id")
    num_aff    = obj.get("num_aff") or key
    num_norm   = norm_num(num_aff)
    body       = (obj.get("texte") or "").strip()
    etat       = "en_vigueur"  # défaut raisonnable

    for idx, piece in enumerate(chunk_text(body)):
        header = f"[CGFP] Article {num_aff} ({art_id}) {num_norm} — "
        full   = header + piece
        rec = {
            "hash_id":     hash_id(CODE_ID, art_id, num_aff, etat, idx),
            "code_id":     CODE_ID,
            "article_id":  art_id,
            "num_article": num_aff,
            "num_norm":    num_norm,
            "role":        "article",
            "chunk_index": idx,
            "text":        full,
            "etat":        etat,
            "date_version": None,
            "source_name": SOURCE,
            "section_path": f"CGFP | art={num_aff} | id={art_id}",
            "thematique":  THEMATIQUE,
        }
        records.append(rec)
        texts_for_embed.append(full)

print(f"→ {len(records)} chunks à embed/injecter")

# ── 🔮 Embeddings (BGE-M3) ─────────────────────────────────────────────────────
emb = model.encode(
    texts_for_embed,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=NORMALIZE
)
# Vérif dimension (BGE-M3 = 1024)
if emb.ndim != 2 or emb.shape[1] != 1024:
    raise ValueError(f"Dim embeddings inattendue: {emb.shape} (attendu: [n,1024])")

# Attacher l'embedding à chaque record
for rec, vec in zip(records, emb):
    rec[EMBED_COL] = vec.tolist()

# ── 🗄️ DDL + UPSERT (création table si besoin) ────────────────────────────────
DDL = f"""
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS law_chunks (
  hash_id         TEXT PRIMARY KEY,
  code_id         TEXT,
  article_id      TEXT NOT NULL,
  num_article     TEXT NOT NULL,
  num_norm        TEXT NOT NULL,
  role            TEXT DEFAULT 'article',
  chunk_index     INT  DEFAULT 0,
  text            TEXT NOT NULL,
  etat            TEXT,
  date_version    DATE,
  source_name     TEXT DEFAULT 'legifrance',
  section_path    TEXT,
  thematique      TEXT,
  {EMBED_COL}     vector(1024),
  created_at      TIMESTAMPTZ DEFAULT NOW()
);

CREATE UNIQUE INDEX IF NOT EXISTS ux_law_article_version_chunk
ON law_chunks (article_id, date_version, chunk_index);

ALTER TABLE law_chunks
  ADD COLUMN IF NOT EXISTS text_tsv tsvector GENERATED ALWAYS AS
  ( to_tsvector('french', coalesce(num_article,'') || ' ' || coalesce(text,'')) ) STORED;

CREATE INDEX IF NOT EXISTS law_text_tsv_idx ON law_chunks USING GIN (text_tsv);

CREATE INDEX IF NOT EXISTS law_embed_idx ON law_chunks
USING ivfflat ({EMBED_COL} vector_cosine_ops) WITH (lists = 200);
"""

UPSERT = f"""
INSERT INTO law_chunks
(hash_id, code_id, article_id, num_article, num_norm, role, chunk_index, text, etat, date_version, source_name, section_path, thematique, {EMBED_COL})
VALUES
(%(hash_id)s, %(code_id)s, %(article_id)s, %(num_article)s, %(num_norm)s, %(role)s, %(chunk_index)s, %(text)s, %(etat)s, %(date_version)s, %(source_name)s, %(section_path)s, %(thematique)s, %({EMBED_COL})s)
ON CONFLICT (article_id, date_version, chunk_index) DO UPDATE
SET
  hash_id      = EXCLUDED.hash_id,
  code_id      = EXCLUDED.code_id,
  num_article  = EXCLUDED.num_article,
  num_norm     = EXCLUDED.num_norm,
  role         = EXCLUDED.role,
  text         = EXCLUDED.text,
  etat         = EXCLUDED.etat,
  source_name  = EXCLUDED.source_name,
  section_path = EXCLUDED.section_path,
  thematique   = EXCLUDED.thematique,
  {EMBED_COL}  = EXCLUDED.{EMBED_COL};
"""

# ── 🚀 Exécution DB ────────────────────────────────────────────────────────────
with psycopg.connect(**PG) as conn:
    with conn.cursor() as cur:
        try:
            cur.execute(DDL)
        except Exception as e:
            print("⚠️  CREATE EXTENSION/INDEX a échoué (droits ?). Suite…", e)

        # We use psycopg.extras.execute_batch for bulk upsert because it efficiently
        # batches parameterized statements into server-side execution. It's faster
        # than looping individual execute() calls and avoids constructing one large
        # multi-row SQL. Note: execute_batch sends the same statement repeatedly with
        # different parameters and relies on the UPSERT (ON CONFLICT DO UPDATE) to
        # ensure idempotency when re-running the notebook.
        
        # Replace execute_batch with a simple runner that executes a list of SQL
        # statements. We render parameterized statements safely using cur.mogrify()
        # so that values are correctly quoted/escaped, then execute the resulting
        # concrete SQL strings.
        def run_sql_list(cur, stmts):
            for s in stmts:
                s = s.strip()
                if s:
                    cur.execute(s)

        # Use executemany for efficient batch upsert in psycopg v3
        cur.executemany(UPSERT, records)
    conn.commit()

print("✅ Ingestion terminée : law_chunks (chunks + embeddings BGE-M3 normalisés).")
print("Astuce test (psql) :")
print("  SELECT num_article, chunk_index, left(text, 140)||'…' AS preview")
print("  FROM law_chunks WHERE num_norm='R331-13' ORDER BY chunk_index;")